In [ ]:
#  Libraries
import os
import glob
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from tqdm import tqdm
import kagglehub
from IPython.display import HTML, display ,Video


"Processing and Converting Caltech Pedestrian Dataset

In [ ]:
class CaltechAnnotationProcessor:
    def __init__(self):
        print("Initializing Processor...")
        self.dataset_path = kagglehub.dataset_download("kalvinquackenbush/caltechpedestriandataset")
        print(f"Dataset downloaded to: {self.dataset_path}")
        self.annotations_path = os.path.join(self.dataset_path, "annotations", "annotations", "*")
        self.classes = ['person']
        self.img_ids = []
        self.labels = []
        self.splits = []
        self.num_annots = []
        self.total_boxes = 0

    def convert_to_yolo(self, box):
        x_left, y_top, width, height = box
        dw = 1. / 640
        dh = 1. / 480
        x = (x_left + width / 2) * dw
        y = (y_top + height / 2) * dh
        w = width * dw
        h = height * dh
        return (x, y, w, h)

    def process(self):
        print(f"Processing annotations from: {self.annotations_path}")
        for annotation_dir in tqdm(sorted(glob.glob(self.annotations_path))):
            print(f"Processing directory: {annotation_dir}")
            set_id = os.path.basename(annotation_dir)
            set_num = int(set_id.replace("set", ""))
            split = "train" if set_num <= 5 else "val"

            for vbb_file in sorted(glob.glob(os.path.join(annotation_dir, "*.vbb"))):
                print(f"Processing VBB file: {vbb_file}")
                video_id = os.path.splitext(os.path.basename(vbb_file))[0]
                data = loadmat(vbb_file)
                obj_list = data['A'][0][0][1][0]
                labels_list = [str(label[0]) for label in data['A'][0][0][4][0]]

                for frame_idx, frame_objs in enumerate(obj_list):
                    if len(frame_objs) == 0:
                        continue
                    frame_labels = ''
                    count = 0
                    for pid, pos in zip(frame_objs['id'][0], frame_objs['pos'][0]):
                        person_id = int(pid[0][0]) - 1
                        bbox = pos[0].tolist()
                        if labels_list[person_id] in self.classes and 75 <= bbox[3] <= 250:
                            yolo = self.convert_to_yolo(bbox)
                            frame_labels += '0 ' + ' '.join(map(str, yolo)) + '\n'
                            count += 1
                            self.total_boxes += 1
                    if frame_labels:
                        img_id = f"{set_id}_{video_id}_{frame_idx:04d}"
                        self.img_ids.append(img_id)
                        self.labels.append(frame_labels)
                        self.splits.append(split)
                        self.num_annots.append(count)

    def save_csv(self, save_path="csv_files/frame_metadata.csv"):
        print(f"Saving CSV to: {save_path}")
        df = pd.DataFrame({
            "image_id": self.img_ids,
            "label": self.labels,
            "split": self.splits,
            "num_annot": self.num_annots
        })
        df["set_id"] = df["image_id"].apply(lambda x: x.split("_")[0])
        df["video_id"] = df["image_id"].apply(lambda x: x.split("_")[1])
        df["frame_id"] = df["image_id"].apply(lambda x: int(x.split("_")[2]))
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        df.to_csv(save_path, index=False)
        return df

    def get_all_paths(self):
        print(f"Getting all paths from: {self.dataset_path}")
        all_paths = []
        for dirpath, dirnames, filenames in os.walk(self.dataset_path):
            all_paths.append(dirpath)
            for filename in filenames:
                all_paths.append(os.path.join(dirpath, filename))
        return all_paths

processor = CaltechAnnotationProcessor()
processor.process()
df = processor.save_csv()
print(f"Total Bounding Boxes: {processor.total_boxes}")
all_paths = processor.get_all_paths()
print("\nAll File Paths:")
for path in all_paths:
    print(path)
print("\nData Summary:")
print(df)

"Filtering and Organizing Caltech Pedestrian Dataset for Training and Validation"



In [ ]:
class ImageFilter:
    def __init__(self, frame_data_sheet, video_group_info, max_train_frames=None, max_val_frames=None, focus_labels=None, output_dir="csv_files"):
        self.frame_data_sheet = frame_data_sheet
        self.video_group_info = video_group_info
        self.max_train_frames = max_train_frames
        self.max_val_frames = max_val_frames
        self.focus_labels = focus_labels if focus_labels is not None else []
        self.output_dir = output_dir

    def filter_images(self):
        train_pool = self.frame_data_sheet[self.frame_data_sheet['split'] == 'train']
        val_pool = self.frame_data_sheet[self.frame_data_sheet['split'] == 'val']

        if self.focus_labels:
            train_pool = train_pool[train_pool['label'].apply (lambda lbl: any(l.split()[0] in self.focus_labels for l in lbl.strip().split('\n')))]
            val_pool = val_pool[val_pool['label'].apply (lambda lbl: any(l.split()[0] in self.focus_labels for l in lbl.strip().split('\n')))]

        if self.max_train_frames is None:
            self.max_train_frames = len(train_pool)

        if self.max_val_frames is None:
            self.max_val_frames = len(val_pool)

        train_ready_set = train_pool.head(self.max_train_frames)
        val_ready_set = val_pool.head(self.max_val_frames)

        return train_ready_set, val_ready_set

    def display_data_summary(self):
        train_pool = self.frame_data_sheet[self.frame_data_sheet['split'] == 'train']
        val_pool = self.frame_data_sheet[self.frame_data_sheet['split'] == 'val']

        total_train = len(train_pool)
        total_val = len(val_pool)

        print(f"Total Training Images: {total_train}")
        print(f"Total Validation Images: {total_val}")

        print("\nFiltered Training Data:")
        display(train_pool.head())

        print("\nFiltered Validation Data:")
        display(val_pool.head())

        print("\nNumber of Videos per Set (Training + Validation):")
        video_distribution = self.video_group_info.groupby("set_id")["video_id"].count().reset_index()
        video_distribution = video_distribution.rename(columns={"video_id": "total_videos"})
        display(video_distribution)

        train_file_path = os.path.join(self.output_dir, "train_frame_filtered.csv")
        val_file_path = os.path.join(self.output_dir, "val_frame_filtered.csv")

        print(f"\nSaved Filtered Training Data to: {train_file_path}")
        print(f"Saved Filtered Validation Data to: {val_file_path}")

image_filter = ImageFilter(
    frame_data_sheet=df,
    video_group_info=df[['set_id', 'video_id']].drop_duplicates(),
    max_train_frames=500,
    max_val_frames=200,
    focus_labels=['0'],
    output_dir="csv_files"
)

train_ready_set, val_ready_set = image_filter.filter_images()

image_filter.display_data_summary()

os.makedirs("csv_files", exist_ok=True)
train_ready_set.to_csv("csv_files/train_frame_filtered.csv", index=False)
val_ready_set.to_csv("csv_files/val_frame_filtered.csv", index=False)

"Filtering Video Frames for Training and Validation Sets in Caltech Pedestrian Dataset"

In [ ]:
class VideoFrameFilter:
    def __init__(self, frame_data_sheet, video_group_info, train_image_total, val_image_total):
        self.frame_data_sheet = frame_data_sheet
        self.video_group_info = video_group_info
        self.train_image_total = train_image_total
        self.val_image_total = val_image_total

    def filter_frames(self):
        train_selection = pd.DataFrame()
        val_selection = pd.DataFrame()
        all_sets = list(self.frame_data_sheet["set_id"].unique())

        for i, current_set in enumerate(all_sets):
            set_videos = self.video_group_info[self.video_group_info["set_id"] == current_set].reset_index(drop=True)
            video_ids = list(set_videos["video_id"].unique())

            for j, vid in enumerate(video_ids):
                vid_frames = self.frame_data_sheet[
                    (self.frame_data_sheet["set_id"] == current_set) &
                    (self.frame_data_sheet["video_id"] == vid)
                ].reset_index(drop=True)

                total_vid_frames = vid_frames.shape[0]

                if i <= 5:
                    limit = max(1, int(round((total_vid_frames / self.train_image_total) * 10000, 0)))
                    selected_frames = vid_frames[:limit]
                    train_selection = pd.concat([train_selection, selected_frames])
                    print(f" Training | Set: {current_set} | Video: {vid} | Frames Selected: {limit}")
                else:
                    limit = max(1, int(round((total_vid_frames / self.val_image_total) * 2500, 0)))
                    selected_frames = vid_frames[:limit]
                    val_selection = pd.concat([val_selection, selected_frames])
                    print(f" Validation | Set: {current_set} | Video: {vid} | Frames Selected: {limit}")

        train_selection = train_selection.reset_index(drop=True)
        val_selection = val_selection.reset_index(drop=True)

        return train_selection, val_selection

    def save_filtered_data(self, train_selection, val_selection):
        os.makedirs("ready_for_training", exist_ok=True)

        train_file_path = "ready_for_training/train_frame_filtered.csv"
        val_file_path = "ready_for_training/val_frame_filtered.csv"

        train_selection.to_csv(train_file_path, index=False)
        val_selection.to_csv(val_file_path, index=False)

        print(f"Saved Training Data to: {train_file_path}")
        print(f"Saved Validation Data to: {val_file_path}")

    def display_filtered_data(self, train_selection, val_selection):
        print("\nFirst pictures from the training set:")
        display(train_selection.head())
        print("\nFirst pictures from the validation set:")
        display(val_selection.head())

df_set_video = df[['set_id', 'video_id']].drop_duplicates()
video_filter = VideoFrameFilter(
    frame_data_sheet=df,
    video_group_info=df_set_video,
    train_image_total=18041,
    val_image_total=12025
)
train_selection, val_selection = video_filter.filter_frames()
df_train_filtered = train_selection.reset_index(drop=True)
df_val_filtered = val_selection.reset_index(drop=True)

video_filter.display_filtered_data(df_train_filtered, df_val_filtered)
video_filter.save_filtered_data(df_train_filtered, df_val_filtered)

"Video to Image Conversion for Caltech Pedestrian Dataset with Annotations"

In [ ]:
class VideoToImageConverter:
    def __init__(self, input_directory, output_directory, split, annotations_df):
        self.input_directory = input_directory
        self.output_directory = output_directory
        self.split = split
        self.annotations_df = annotations_df

    def save_img(self, dir_path, fn, frame_index, frame):
        img_filename = f'{dir_path}/{os.path.basename(dir_path)}_{os.path.basename(fn).split(".")[0]}_{frame_index:04d}.png'
        cv2.imwrite(img_filename, frame)

    def convert_videos_to_images(self):
        print(f" Starting image conversion for: {self.split}")
        output_dir = os.path.join(self.output_directory, self.split.lower(), "caltechpedestriandataset")
        os.makedirs(output_dir, exist_ok=True)
        sets_list = sorted(glob.glob(os.path.join(self.input_directory, self.split, "set*")))
        print(f" Number of Sets: {len(sets_list)}")

        for set_path in sets_list:
            set_name = os.path.basename(set_path)
            output_dir_set = os.path.join(output_dir, set_name)
            os.makedirs(output_dir_set, exist_ok=True)

            filtered_df = self.annotations_df[self.annotations_df["set_id"] == set_name].reset_index(drop=True)
            video_ids = filtered_df["video_id"].unique()
            print(f"\n Set: {set_name} | Videos: {len(video_ids)}")

            for video_id in video_ids:
                video_path = os.path.join(set_path, set_name, f"{video_id}.seq")
                print(f" Processing: {video_path}")

                cap = cv2.VideoCapture(video_path)
                if not cap.isOpened():
                    print(f" Failed to open {video_path}")
                    continue

                filtered_video_df = filtered_df[filtered_df["video_id"] == video_id]
                frame_ids = set(filtered_video_df["frame_id"].unique())
                print(f" Frames to save: {len(frame_ids)}")

                frame_index = 0
                saved_frames = 0

                while True:
                    ret, frame = cap.read()
                    if not ret:
                        break
                    if frame_index in frame_ids:
                        self.save_img(output_dir_set, video_path, frame_index, frame)
                        saved_frames += 1
                        if saved_frames == len(frame_ids):
                            break
                    frame_index += 1

                cap.release()
                print(f" Done: {video_id} | Saved: {saved_frames} frames")

original_data = processor.dataset_path
after_convert_data = "datasets/images"
df_annotations = df

train_converter = VideoToImageConverter(original_data, after_convert_data, "Train", df_annotations)
train_converter.convert_videos_to_images()

test_converter = VideoToImageConverter(original_data, after_convert_data, "Test", df_annotations)
test_converter.convert_videos_to_images()

 "First Frame from Each Video in set05"

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

class LabelWriter:
    def __init__(self, base_path="datasets/labels"):
        self.base_path = base_path

    def write_labels(self, phase, data_df):
        save_path = os.path.join(self.base_path, phase.lower())
        os.makedirs(save_path, exist_ok=True)
        print(f"Created or found folder: {save_path}")

        all_sets = data_df["set_id"].unique()

        for set_name in all_sets:
            set_folder = os.path.join(save_path, set_name)
            os.makedirs(set_folder, exist_ok=True)
            print(f"Created or found folder for Set: {set_name}")

            set_rows = data_df[data_df["set_id"] == set_name].reset_index(drop=True)

            for i, row in set_rows.iterrows():
                file_path = os.path.join(set_folder, row["image_id"] + ".txt")
                with open(file_path, 'w') as f:
                    f.write(str(row["label"]))

            print(f"Labels written for Set: {set_name} | Total: {len(set_rows)} labels")

class FullSetConverter:
    def __init__(self, seq_dir, output_dir):
        self.seq_dir = seq_dir
        self.output_dir = output_dir

    def convert_set(self, set_id):
        set_path = os.path.dirname(self.seq_dir)    # Use the original seq_dir instead of joining it again
        print(f"Found file {set_path}:", os.listdir(set_path))

        video_files = sorted([f for f in os.listdir(set_path) if f.endswith(".seq")])

        print(f"Start converting set: {set_id} | Total Videos: {len(video_files)}")

        for video_file in video_files:
            video_id = video_file.split(".")[0]
            input_video_path = os.path.join(set_path, video_file)
            save_dir = os.path.join(self.output_dir, set_id)
            os.makedirs(save_dir, exist_ok=True)

            cap = cv2.VideoCapture(input_video_path)
            if not cap.isOpened():
                print(f"Failed to open {input_video_path}")
                continue

            frame_idx = 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                image_name = f"{set_id}_{video_id}_{frame_idx:04d}.png"
                image_path = os.path.join(save_dir, image_name)
                cv2.imwrite(image_path, frame)
                frame_idx += 1

            cap.release()
            print(f"Converted {video_id} | Total Frames: {frame_idx}")

class SetImageVisualizer:
    def __init__(self, image_base_path):
        self.image_base_path = image_base_path

    def show_first_images(self, set_id, num_videos=6):
        set_path = os.path.join(self.image_base_path, set_id)
        if not os.path.exists(set_path):
            print(f"Path not found: {set_path}")
            return

        image_files = sorted([f for f in os.listdir(set_path) if f.endswith(".png")])
        video_ids = sorted(list(set([f.split("_")[1] for f in image_files])))

        print(f"Showing first frames for {len(video_ids)} videos")

        n_cols = 3
        n_rows = min((num_videos + 2) // 3, 3)
        fig, axs = plt.subplots(n_rows, n_cols, figsize=(12, 10))

        idx = 0
        for i in range(n_rows):
            for j in range(n_cols):
                if idx >= len(video_ids) or idx >= num_videos:
                    axs[i, j].axis("off")
                    continue

                vid = video_ids[idx]
                frame_name = f"{set_id}_{vid}_0000.png"
                frame_path = os.path.join(set_path, frame_name)

                img = cv2.imread(frame_path)
                if img is not None:
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    axs[i, j].imshow(img)
                    axs[i, j].axis("off")
                    axs[i, j].set_title(f"{vid}")
                else:
                    axs[i, j].text(0.5, 0.5, "", fontsize=20)
                    axs[i, j].axis("off")

                idx += 1

        plt.tight_layout()
        plt.suptitle(f"First Frame from Each Video in {set_id}", fontsize=16)
        plt.show()

converter = FullSetConverter(
    seq_dir="/kaggle/input/caltechpedestriandataset/Train/set05/set05/V008.seq"
,
    output_dir="datasets/images/train"
)

converter.convert_set("set05")

viewer = SetImageVisualizer("datasets/images/train")
viewer.show_first_images("set05", num_videos=9)



Segmenting Frames and Converting to Video for Caltech Pedestrian Dataset

In [ ]:
class FrameSegmenter:
    def __init__(self, dataset_path, set_num):
        self.dataset_path = dataset_path
        self.set_num = set_num
        self.set_path = f"{dataset_path}/set{set_num:02d}"

        if not os.path.exists(self.set_path):
            raise FileNotFoundError(f" Path not found: {self.set_path}")

        self.frames = sorted([
            f for f in os.listdir(self.set_path) if f.endswith((".jpg", ".png"))
        ])
        if not self.frames:
            raise ValueError(" No frames found in the folder!")

        self.frame_segment_list = []
        self.segment_frames()

    def segment_frames(self):
        frame_segment = []
        check_segment = False

        for i in range(len(self.frames)):
            parts = self.frames[i].split("_")
            try:
                num_frame = int(parts[-1][:4])
            except:
                continue

            if not check_segment or num_frame - 1 == frame_segment[-1]:
                check_segment = True
                frame_segment.append(num_frame)
            else:
                self.frame_segment_list.append({
                    "video_id": parts[1],
                    "frames": frame_segment.copy()
                })
                frame_segment = [num_frame]
                check_segment = False

            if len(self.frame_segment_list) == 10:
                break

        print(f" Number of frame groups: {len(self.frame_segment_list)}")

    def get_frames_from_segment(self, segment_index):
        segment = self.frame_segment_list[segment_index]
        video_id = segment["video_id"]
        return [
            f"{self.set_path}/set{self.set_num:02d}_{video_id}_{x:04d}.png"
            for x in segment["frames"]
        ]

    def convert_frames_to_video(self, frame_list, output):
        fps = 25.0
        frame_size = (640, 480)
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        video = cv2.VideoWriter(output, fourcc, fps, frame_size)

        valid_frames = 0
        for frame in frame_list:
            img = cv2.imread(frame)
            if img is None:
                print(f" Skipping bad frame: {frame}")
                continue
            img = cv2.resize(img, frame_size)
            video.write(img)
            valid_frames += 1

        video.release()
        print(f" Saved video with {valid_frames} frames → {output}")

dataset_path = "datasets/images/train/caltechpedestriandataset"
set_num = 1

frame_segmenter = FrameSegmenter(dataset_path, set_num)

frame_segment_choice = frame_segmenter.get_frames_from_segment(2)
print(f" Number of frames in selected segment: {len(frame_segment_choice)}")

os.makedirs("sample_videos", exist_ok=True)
output_mp4 = "sample_videos/sample_video_1.mp4"
frame_segmenter.convert_frames_to_video(frame_segment_choice, output_mp4)

!ffmpeg -y -i sample_videos/sample_video_1.mp4 sample_videos/sample_video_1.webm

Video("sample_videos/sample_video_1.webm", embed=True)

Augmentation

In [ ]:
import albumentations as A

class Augmenter:
    def __init__(self, images_dir, labels_df, output_dir, num_augmented=1):
        self.images_dir = images_dir
        self.labels_df = labels_df
        self.output_dir = output_dir
        self.num_augmented = num_augmented

        self.transform = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.4),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.4),
            A.GaussianBlur(blur_limit=(3, 5), p=0.3)
        ])

        os.makedirs(self.output_dir, exist_ok=True)

    def apply_augmentation(self):
        new_rows = []

        for i, row in tqdm(self.labels_df.iterrows(), total=len(self.labels_df)):
            image_id = row['image_id']
            label = row['label']
            set_id = row['set_id']

            image_path = os.path.join(self.images_dir, set_id, f"{image_id}.png")
            if not os.path.exists(image_path):
                print(f" Missing image: {image_path}")
                continue

            image = cv2.imread(image_path)
            if image is None:
                print(f" Bad image: {image_path}")
                continue

            for aug_id in range(self.num_augmented):
                augmented = self.transform(image=image)['image']
                new_image_id = f"{image_id}_aug{aug_id}"
                new_filename = f"{new_image_id}.png"
                new_path = os.path.join(self.output_dir, set_id)
                os.makedirs(new_path, exist_ok=True)
                cv2.imwrite(os.path.join(new_path, new_filename), augmented)

                new_rows.append({
                    "image_id": new_image_id,
                    "label": label,
                    "split": "train",
                    "num_annot": row['num_annot'],
                    "set_id": set_id,
                    "video_id": row['video_id'],
                    "frame_id": row['frame_id']
                })

        print(f"\n Augmentation done! Total new images: {len(new_rows)}")
        return pd.DataFrame(new_rows)

train_df = df[df["split"] == "train"].reset_index(drop=True)
original_images_dir = "datasets/images/train/caltechpedestriandataset"
augmented_output_dir = "datasets/images/augmented_train"

augmenter = Augmenter(
    images_dir=original_images_dir,
    labels_df=train_df,
    output_dir=augmented_output_dir,
    num_augmented=2
)

augmented_df = augmenter.apply_augmentation()
train_total = pd.concat([train_df, augmented_df], ignore_index=True)
train_total.to_csv("csv_files/train_frame_augmented.csv", index=False)

Data Processor Train ,Test

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import os
from IPython.display import display

class DataProcessorTrainTest:
    def __init__(self, train_metadata_path, test_metadata_path):
        self.df_train_metadata = pd.read_csv(train_metadata_path)
        self.df_test_metadata = pd.read_csv(test_metadata_path)
        self.df_train_annot = self.get_annot_dataframe(self.df_train_metadata)
        self.df_test_annot = self.get_annot_dataframe(self.df_test_metadata)
        self.df_train_var = self.get_variation_df(self.df_train_metadata, self.df_train_annot)
        self.df_test_var = self.get_variation_df(self.df_test_metadata, self.df_test_annot)

    def convert_yolo_to_coco(self, bbox):
        x_center, y_center, width, height = bbox
        x1 = x_center - (width / 2)
        y1 = y_center - (height / 2)
        x2 = x_center + (width / 2)
        y2 = y_center + (height / 2)
        return [x1, y1, x2, y2]

    def get_annot_dataframe(self, df):
        image_id_list = []
        label_list = []
        for i in range(len(df)):
            image_id = df["image_id"][i]
            labels = df["label"][i].split("\n")[:-1]
            for j in range(len(labels)):
                list_box = [float(x) for x in labels[j][:-1].split(" ")][1:]
                list_box = self.convert_yolo_to_coco(list_box)
                image_id_list.append(image_id)
                label_list.append(list_box)
        df_annot = pd.DataFrame({"image_id": image_id_list, "label": label_list})
        df_annot["x1"] = df_annot["label"].apply(lambda x: x[0])
        df_annot["y1"] = df_annot["label"].apply(lambda x: x[1])
        df_annot["x2"] = df_annot["label"].apply(lambda x: x[2])
        df_annot["y2"] = df_annot["label"].apply(lambda x: x[3])
        df_annot["p_width"] = df_annot["x2"] - df_annot["x1"]
        df_annot["p_height"] = df_annot["y2"] - df_annot["y1"]
        df_annot["area"] = df_annot["p_width"] * df_annot["p_height"]
        df_annot["aspect_ratio"] = df_annot["p_width"] / df_annot["p_height"]
        return df_annot.drop("label", axis=1)

    def get_variation_df(self, df_met, df_annot):
        df_met = df_met[df_met["num_annot"] > 1].reset_index(drop=True)
        ids = df_met["image_id"].unique()
        df_annot = df_annot[df_annot["image_id"].isin(ids)]

        df_grouped = df_annot.groupby("image_id").agg({
            "p_width": ["min", "max"],
            "p_height": ["min", "max"],
            "area": ["min", "max"],
            "aspect_ratio": ["min", "max"]
        }).reset_index()
        df_grouped.columns = ['_'.join(col).strip('_') for col in df_grouped.columns]

        df_grouped["p_width_range"] = df_grouped["p_width_max"] - df_grouped["p_width_min"]
        df_grouped["p_height_range"] = df_grouped["p_height_max"] - df_grouped["p_height_min"]
        df_grouped["area_range"] = df_grouped["area_max"] - df_grouped["area_min"]
        df_grouped["aspect_ratio_range"] = df_grouped["aspect_ratio_max"] - df_grouped["aspect_ratio_min"]

        return df_grouped[["image_id", "p_width_range", "p_height_range", "area_range", "aspect_ratio_range"]]

    def plot_people_distribution_with_stats(self):
        df_train_num_annot = self.df_train_metadata.groupby("num_annot")["image_id"].count().reset_index()
        df_train_num_annot = df_train_num_annot.rename(columns={"image_id": "total_image"})

        df_test_num_annot = self.df_test_metadata.groupby("num_annot")["image_id"].count().reset_index()
        df_test_num_annot = df_test_num_annot.rename(columns={"image_id": "total_image"})

        print(f"People count in train images:")
        print(df_train_num_annot)
        print(f"\nPeople count in test images:")
        print(df_test_num_annot)

        fig = make_subplots(rows=1, cols=2, subplot_titles=["Training", "Test"])

        fig.add_trace(go.Bar(x=df_train_num_annot["num_annot"], y=df_train_num_annot["total_image"],
                             marker_color="#9f424d", name="Train"), row=1, col=1)
        fig.add_trace(go.Bar(x=df_test_num_annot["num_annot"], y=df_test_num_annot["total_image"],
                             marker_color="#4083d9", name="Test"), row=1, col=2)

        fig.update_layout(title="People Distribution per Frame", title_x=0.5, width=950, height=400)
        fig.show()

        self.people_summary()  # Call the extra summary

    def people_summary(self):
        print("\n Summary of People per Frame:\n")
        for dataset_name, df in [("Train", self.df_train_metadata), ("Test", self.df_test_metadata)]:
            total_images = df.shape[0]
            total_people = df["num_annot"].sum()
            mean_people = df["num_annot"].mean()
            max_people = df["num_annot"].max()
            more_than_5 = (df["num_annot"] > 5).sum()
            zero_people = (df["num_annot"] == 0).sum()

            print(f" Dataset: {dataset_name}")
            print(f" Total Images: {total_images}")
            print(f" Total People: {total_people}")
            print(f" Avg People per Image: {mean_people:.2f}")
            print(f" Max People in One Image: {max_people}")
            print(f" Images with > 5 People: {more_than_5}")
            print(f" Images with 0 People: {zero_people}\n")

    def plot_distribution_with_stats(self, train_df, test_df, param, title):
        train_mean = train_df[param].mean()
        test_mean = test_df[param].mean()
        train_std = train_df[param].std()
        test_std = test_df[param].std()

        print(f"\n{title} - Training Data")
        print(f"  Mean: {train_mean:.2f}, Std: {train_std:.2f}")
        print(f"\n{title} - Test Data")
        print(f"  Mean: {test_mean:.2f}, Std: {test_std:.2f}")

        summary_df = pd.DataFrame({
            "Dataset": ["Train", "Test"],
            "Mean": [train_mean, test_mean],
            "Std Dev": [train_std, test_std],
            "Min": [train_df[param].min(), test_df[param].min()],
            "Max": [train_df[param].max(), test_df[param].max()],
            "Median": [train_df[param].median(), test_df[param].median()],
            "25%": [train_df[param].quantile(0.25), test_df[param].quantile(0.25)],
            "75%": [train_df[param].quantile(0.75), test_df[param].quantile(0.75)]
        })

        print("\n Summary Table:")
        display(summary_df)

        if len(train_df[param].dropna()) < 2 or len(test_df[param].dropna()) < 2:
            print(f"Warning: Insufficient data points for creating distribution plot for {param}. Skipping...")
            return

        if train_df[param].isnull().any() or test_df[param].isnull().any():
            print(f"Warning: Columns {param} contain null values.")
            return

        fig = make_subplots(rows=1, cols=2, subplot_titles=[f"{title} Distribution", f"{title} Range"])

        fig_dist = ff.create_distplot(
            [train_df[param].dropna(), test_df[param].dropna()],
            ["Train", "Test"], show_rug=False
        )
        fig.add_trace(go.Scatter(fig_dist['data'][2], line=dict(color="#9f424d")), row=1, col=1)
        fig.add_trace(go.Scatter(fig_dist['data'][3], line=dict(color="#4083d9")), row=1, col=1)

        if len(self.df_train_var[f"{param}_range"].dropna()) >= 2 and len(self.df_test_var[f"{param}_range"].dropna()) >= 2:
            fig_range = ff.create_distplot(
                [self.df_train_var[f"{param}_range"].dropna(), self.df_test_var[f"{param}_range"].dropna()],
                ["Train", "Test"], show_rug=False
            )
            fig.add_trace(go.Scatter(fig_range['data'][2], line=dict(color="#9f424d")), row=1, col=2)
            fig.add_trace(go.Scatter(fig_range['data'][3], line=dict(color="#4083d9")), row=1, col=2)

        fig.update_layout(title=title, title_x=0.5, width=950, height=400)
        fig.show()
df_test = df[df["split"] == "val"].reset_index(drop=True)
os.makedirs("csv_files", exist_ok=True)
df_test.to_csv("csv_files/test_frame_filtered.csv", index=False)

data_processor = DataProcessorTrainTest(
    "csv_files/train_frame_augmented.csv",
    "csv_files/test_frame_filtered.csv"
)

data_processor.plot_people_distribution_with_stats()

params = ["p_width", "p_height", "area", "aspect_ratio"]
titles = ["Width", "Height", "Area", "Aspect Ratio"]

for param, title in zip(params, titles):
    data_processor.plot_distribution_with_stats(
        data_processor.df_train_annot, data_processor.df_test_annot, param, title
    )


In [ ]:
TRAIN_IMG_DIR = "datasets/images/train/caltechpedestriandataset"
VAL_IMG_DIR = "datasets/images/val/caltechpedestriandataset"
AUG_IMG_DIR = "datasets/images/augmented_train"

TRAIN_LABEL_DIR = "datasets/labels/train"
VAL_LABEL_DIR = "datasets/labels/val"

TRAIN_CSV = "csv_files/train_frame_augmented.csv"
VAL_CSV = "csv_files/test_frame_filtered.csv"

VIDEO_SAMPLE = "sample_videos/sample_video_1.mp4"

YOLO_DATA_YAML = "yolov5/caltech.yaml"

FASTER_MODEL_PATH = "models/faster_rcnn_trained.pth"

In [ ]:
!pip install ultralytics

Model Training and evalution 

In [ ]:
import os

# Create the directory if it doesn't exist
os.makedirs("/kaggle/working/", exist_ok=True)

# Write content to the file
file_path = "/kaggle/working/custom_dataset.yaml"

print(f"Writing to {file_path}")  # Debugging line

with open(file_path, "w") as f:
    f.write("""
# Create Custom Dataset Configuration
path: "../datasets/images"  # Updated path to your data set
train: "train/caltechpedestriandataset/train_frame_augmented.csv"  # Updated path to train CSV
val: "val/caltechpedestriandataset/test_frame_filtered.csv"  # Updated path to val CSV

nc: 1

names: [
    'person'
]
""")

Writing to /kaggle/working/custom_dataset.yaml


In [ ]:
!wandb disabled
!pip install GPUtil

W&B disabled.


In [ ]:
from GPUtil import showUtilization as gpu_usage
import torch
from numba import cuda

def free_gpu_cache():
    print("Initial GPU Usage")
    gpu_usage()

    torch.cuda.empty_cache()

    cuda.select_device(0)
    cuda.close()
    cuda.select_device(0)

    print("GPU Usage after emptying the cache")
    gpu_usage()

free_gpu_cache()

Initial GPU Usage
| ID | GPU | MEM |
------------------
|  0 |  0% |  1% |
GPU Usage after emptying the cache
| ID | GPU | MEM |
------------------
|  0 |  1% |  1% |


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
epochs = 10
model.train(data="/kaggle/working/custom_dataset.yaml", epochs=epochs, verbose=True, batch=64)

Model detection

In [ ]:
from ultralytics import YOLO
import cv2
import os
from streamlit_webrtc import VideoTransformerBase

model = YOLO("weights/best.pt")
model_2 = YOLO("weights/yolov8n.pt")

def detect_image(image_path, output_path):
    results = model_2.predict(image_path, save=False, conf=0.5,classes=[0])
    frame = cv2.imread(image_path)

    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            if conf >= 0.5:
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                label = f"Conf: {conf:.2f}"
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    cv2.imwrite(output_path, frame)
    return output_path

class LiveCamProcessor(VideoTransformerBase):
    def transform(self, frame):
        img = frame.to_ndarray(format="bgr24")
        results = model_2.predict(img, conf=0.4, save=False,classes=[0])

        for result in results:
            for box in result.boxes:
                cls_id = int(box.cls[0])
                if cls_id != 0:  #  الفئة 0 = person
                    continue

                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                if conf >= 0.4:
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    label = f"{conf:.2f}"
                    cv2.putText(img, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        return img

def detect_video(video_path, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    width  = int(cap.get(3))
    height = int(cap.get(4))
    fps = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*'vp80')  # لـ webm
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model.predict(frame, save=False, conf=0.5,classes=[0])
        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                if conf >= 0.5:
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    label = f"Conf: {conf:.2f}"
                    cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

        out.write(frame)

    cap.release()
    out.release()
    print("Video saved to:", output_path)
    return output_path

Appliction 

In [ ]:
#######To run the app write in terminal "streamlit run app.py"##########################

import streamlit as st
import os
import time
import uuid
from yolov8_inference import detect_image, detect_video, LiveCamProcessor
from streamlit_webrtc import webrtc_streamer
import base64

# Set page config with modern theme and logo
st.set_page_config(
    page_title="VisionWalk",
    layout="centered",
    page_icon="Logo.png" # Make sure to have Logo.png in your directory
)

# Function to encode image to base64
def get_image_base64(image_path):
    with open(image_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode('utf-8')

# Function to set background image
def set_background(image_path):
    with open(image_path, "rb") as img_file:
        encoded_string = base64.b64encode(img_file.read()).decode()
    
    st.markdown(
        f"""
        <style>
        .stApp {{
            background-image: url(data:image/jpg;base64,{encoded_string});
            background-size: cover;
            background-attachment: fixed;
            background-position: center;
            background-blur: 2px;
        }}
        </style>
        """,
        unsafe_allow_html=True
    )

# Set background image (replace with your actual image path)
background_image_path = "Background.png"
if os.path.exists(background_image_path):
    set_background(background_image_path)

# Custom CSS for the modern, youthful design
st.markdown(
    """
    <style>
    @keyframes float {
        0% { transform: translateY(0px); }
        50% { transform: translateY(-10px); }
        100% { transform: translateY(0px); }
    }
    
    @keyframes gradientBG {
        0% { background-position: 0% 50%; }
        50% { background-position: 100% 50%; }
        100% { background-position: 0% 50%; }
    }
    
    :root {
        --primary: #ff9ff3;
        --secondary: #a55eea;
        --dark: #341f97;
        --light: #fef6e4;
    }
    
    body {
        background-image: url('Background.png');
        background-size: cover;
        background-attachment: fixed;
        background-position: center;
        color: white;
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        margin: 0;
        padding: 0;
    }
    
    .stApp {
        background-color: rgba(255, 255, 255, 0.15);
        backdrop-filter: blur(4px);
        min-height: 100vh;
    }
    
    .logo-container {
        position: absolute;
        top:-30px;
        left:-10px;
        z-index: 1000;
        animation: float 6s ease-in-out infinite;
    }
    
    .logo {
        width: 80px;
        height: auto;
        border-radius: 50%;
        box-shadow: 0 10px 30px rgba(165, 94, 234, 0.4);
        transition: all 0.3s ease;
    }
    
    .logo:hover {
        transform: scale(1.05) rotate(5deg);
        box-shadow: 0 15px 40px rgba(165, 94, 234, 0.6);
    }
    
    .title {
        text-align: center;
        font-size: 2.8em;
        background: white;
        -webkit-background-clip: text;
        background-clip: text;
        color: transparent;
        margin-bottom: 0.3em;
        font-weight: 800;
        text-shadow: 2px 2px 4px rgba(0,0,0,0.1);
        animation: float 4s ease-in-out infinite;
    }
    
    .section-title {
        background: rgba(255, 255, 255, 0.7);
        padding: 12px 20px;
        border-radius: 20px;
        background: linear-gradient(to right, #ffd6e7, #e2c1ff);
        -webkit-background-clip: text;
        background-clip: text;
        color: transparent;
        margin-top: 1.5em;
        font-size: 1.4em;
        font-weight: 600;
        box-shadow: 0 4px 15px rgba(0,0,0,0.1);
        border-left: 5px solid #a55eea;
        transition: all 0.3s ease;
        margin-bottom: 1em;
    }
    
    .section-title:hover {
        transform: translateY(-3px);
        box-shadow: 0 8px 25px rgba(0,0,0,0.15);
    }
    
    .stRadio > div {
        background: rgba(255, 255, 255, 0.7);
        border-radius: 15px;
        padding: 15px;
        box-shadow: 0 4px 15px rgba(0,0,0,0.1);
        transition: all 0.3s ease;
        margin-bottom: 1.5em;
    }
    
    .stRadio > div:hover {
        transform: translateY(-3px);
        box-shadow: 0 8px 25px rgba(0,0,0,0.15);
    }
    
    .stFileUploader {
        background: rgba(255, 255, 255, 0.7);
        border-radius: 15px;
        padding: 20px;
        box-shadow: 0 4px 15px rgba(0,0,0,0.1);
        transition: all 0.3s ease;
        margin-bottom: 1.5em;
    }
    
    .stFileUploader:hover {
        transform: translateY(-3px);
        box-shadow: 0 8px 25px rgba(0,0,0,0.15);
    }
    
    .stButton>button {
        background: linear-gradient(to right, #a55eea, #ff9ff3);
        color: white;
        border: none;
        border-radius: 12px;
        padding: 10px 24px;
        font-weight: 600;
        transition: all 0.3s;
        box-shadow: 0 4px 15px rgba(165, 94, 234, 0.3);
        margin-bottom: 1.5em;
    }
    
    .stButton>button:hover {
        transform: translateY(-3px);
        box-shadow: 0 8px 25px rgba(165, 94, 234, 0.4);
        background: linear-gradient(to right, #ff9ff3, #a55eea);
    }
    
    .stSpinner>div {
        color: #a55eea;
    }
    
    .stAlert {
        border-radius: 15px;
        transition: all 0.3s ease;
        margin-bottom: 1.5em;
        background: rgba(255, 255, 255, 0.7);
        box-shadow: 0 4px 15px rgba(0,0,0,0.1);
    }
    
    .stAlert:hover {
        transform: translateY(-3px);
        box-shadow: 0 8px 25px rgba(0,0,0,0.15);
    }
    
    .stVideo {
        border-radius: 15px;
        overflow: hidden;
        box-shadow: 0 8px 25px rgba(0,0,0,0.15);
        transition: all 0.3s ease;
        margin-bottom: 1.5em;
    }
    
    .stVideo:hover {
        transform: translateY(-5px);
        box-shadow: 0 12px 30px rgba(0,0,0,0.2);
    }
    
    .stImage {
        border-radius: 15px;
        overflow: hidden;
        box-shadow: 0 8px 25px rgba(0,0,0,0.15);
        transition: all 0.3s ease;
        margin-bottom: 1.5em;
    }
    
    .stImage:hover {
        transform: translateY(-5px);
        box-shadow: 0 12px 30px rgba(0,0,0,0.2);
    }
    
    .floating {
        animation: float 6s ease-in-out infinite;
    }
    
    .text-content {
        color: white;
        text-shadow: 1px 1px 3px rgba(0,0,0,0.3);
    }
    
    .gradient-text {
        background: linear-gradient(to right, #ff9ff3, #a55eea);
        -webkit-background-clip: text;
        background-clip: text;
        color: transparent;
    }

    </style>
    """,
    unsafe_allow_html=True
)

# Logo in top-left corner
logo_path = "Logo1.png"
if os.path.exists(logo_path):
    logo_base64 = get_image_base64(logo_path)
    st.markdown(
        f"""
        <div class="logo-container">
            <img src="data:image/png;base64,{logo_base64}" class="logo" alt="VisionWalk Logo">
        </div>
        """, 
        unsafe_allow_html=True
    )

# Header
st.markdown(
    """
    <div class="title">
        <span style="font-size: 1.2em;">✨</span> VisionWalk AI 
        <span style="font-size: 1.2em;">🚶</span><br>
        <span style="font-size: 0.6em; display: block; margin-top: 5px;">
        Real-time Person Detection with YOLOv8
        </span>
    </div>
    """, 
    unsafe_allow_html=True
)

# Short description
st.markdown(
    """
    <div style="text-align: center; margin-bottom: 30px; color: white; font-size: 1.1em;" class="text-content">
    Detect people in images, videos, or live camera feed using cutting-edge AI technology
    </div>
    """,
    unsafe_allow_html=True
)

# --- Section Selector ---
st.markdown('<div class="section-title">📲 Choose Your Detection Method</div>', unsafe_allow_html=True)
option = st.radio(
    "Select input type:", 
    ["Upload Image", "Upload Video", "Use Live Camera"], 
    horizontal=True,
    label_visibility="collapsed"
)

# Divider with style
st.markdown("---")

# --- Uploaded Image ---
if option == "Upload Image":
    st.markdown('<div class="section-title">📸 Image Detection</div>', unsafe_allow_html=True)
    st.markdown("""
    <div class="text-content" style="margin-bottom: 20px;">
    Upload an image to detect people using our AI model
    </div>
    """, unsafe_allow_html=True)
    
    uploaded_image = st.file_uploader(
        "Choose an image...", 
        type=["jpg", "jpeg", "png"],
        label_visibility="collapsed"
    )
    
    if uploaded_image:
        col1, col2 = st.columns(2)
        with col1:
            st.markdown("**<span class='gradient-text'>Original Image</span>**", unsafe_allow_html=True)
            st.image(uploaded_image, use_container_width =True)
        
        with st.spinner("🔍 AI is analyzing your image..."):
            input_path = os.path.join("uploads", uploaded_image.name)
            with open(input_path, "wb") as f:
                f.write(uploaded_image.read())
            
            output_path = os.path.join("outputs", "detected_image.jpg")
            result = detect_image(input_path, output_path)
            
            with col2:
                st.markdown("**<span class='gradient-text'>Detected People</span>**", unsafe_allow_html=True)
                st.image(result, use_container_width =True)
        
        st.success("Detection completed!")

# --- Uploaded Video ---
elif option == "Upload Video":
    st.markdown('<div class="section-title">🎥 Video Detection</div>', unsafe_allow_html=True)
    st.markdown("""
    <div class="text-content" style="margin-bottom: 20px;">
    Upload a video to detect people in motion
    </div>
    """, unsafe_allow_html=True)
    
    uploaded_video = st.file_uploader(
        "Choose a video...", 
        type=["mp4", "avi", "mov", "webm"],
        label_visibility="collapsed"
    )
    
    if uploaded_video:
        st.markdown("**<span class='gradient-text'>Original Video</span>**", unsafe_allow_html=True)
        st.video(uploaded_video)
        
        unique_name = f"detected_{uuid.uuid4().hex}.webm"
        output_path = os.path.join("outputs", unique_name)
        
        with st.spinner("🎬 Processing video with AI (this may take a while)..."):
            input_path = os.path.join("uploads", uploaded_video.name)
            with open(input_path, "wb") as f:
                f.write(uploaded_video.read())
            
            result_path = detect_video(input_path, output_path)
        
        # Wait for file to be ready
        wait_time = 0
        while (not os.path.exists(result_path) or os.path.getsize(result_path) < 1000 and wait_time < 5):
            time.sleep(0.5)
            wait_time += 0.5
        
        if os.path.exists(result_path):
            st.success("✅ Detection completed! Here's the result:")
            st.markdown("**<span class='gradient-text'>Processed Video</span>**", unsafe_allow_html=True)
            with open(result_path, "rb") as video_file:
                video_bytes = video_file.read()
                st.video(video_bytes)
        else:
            st.error("❌ Something went wrong. Please try again.")

# --- Live Camera ---
elif option == "Use Live Camera":
    st.markdown('<div class="section-title">📡 Live Camera Detection</div>', unsafe_allow_html=True)
    
    st.info("""
    **ℹ️ How to use:**  
    - Click 'Start' to begin camera detection  
    - Make sure to allow camera access in your browser  
    - Click 'Stop' when finished  
    """, icon="💡")
    
    # Camera visualization with glassmorphism effect
    st.markdown(
        """
        <style>
        .camera-container {
            background: rgba(255, 255, 255, 0.3);
            backdrop-filter: blur(10px);
            border-radius: 20px;
            padding: 20px;
            box-shadow: 0 8px 32px rgba(165, 94, 234, 0.2);
            border: 1px solid rgba(255, 255, 255, 0.2);
            margin-bottom: 20px;
            transition: all 0.3s ease;
        }
        
        .camera-container:hover {
            transform: translateY(-5px);
            box-shadow: 0 12px 35px rgba(165, 94, 234, 0.3);
        }
        </style>
        
        <div class="camera-container floating">
        """,
        unsafe_allow_html=True
    )
    
    webrtc_streamer(
        key="live",
        video_processor_factory=LiveCamProcessor,
        media_stream_constraints={"video": True, "audio": False},
        async_processing=True,
    )
    
    st.markdown("</div>", unsafe_allow_html=True)